# MIRA-Bench Text Prompt Agent Evaluation

Run the MMEdit agent on MIRA-Bench using the original text instruction, without visual prompt selectors. The dataset is a file-based Hugging Face snapshot, so setup downloads its benchmark images and uses them to resolve the local image references in the annotations. There are no target images. By default, this notebook processes every annotation; set `SAMPLE_LIMIT = 1` for a smoke test. Set `RESUME_LAST_RUN = True` to continue the latest run, skipping completed samples and retrying failed ones.

Install with `uv sync --extra models --extra benchmark`. Annotation records are loaded from the latest `.out/annotation/mira_bench/run_*/mira_bench_mixed.jsonl`; if no canonical run exists, the latest legacy `.out/annotation/mirage/run_*/mirage_mixed.jsonl` is used. Override the run with `MMEDIT_MIRA_BENCH_ANNOTATION_RUN=run_<timestamp>` or the file with `MMEDIT_MIRA_BENCH_ANNOTATIONS`. Configure `MMEDIT_API_KEY` or `GOOGLE_API_KEY`.

Results are saved under `.out/evaluation/mira_bench/text/run_<timestamp>/`.

In [ ]:
import os

from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
from evaluation.notebook import create_run_directory, load_benchmark
from evaluation.common import annotate_sample, load_sample, run_benchmark, select_reference_prompts

DATASET_ID = "ziqiangoodgood/MIRAGE"
DATASET_SNAPSHOT = snapshot_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    allow_patterns=["benchmark/*.jpg", "benchmark/*.jpeg", "benchmark/*.png", "benchmark/*.webp"],
)
os.environ["MIRA_BENCH_IMAGE_ROOT"] = os.path.join(DATASET_SNAPSHOT, "benchmark")

BENCHMARK = "mira_bench"
PROMPT_MODE = "text"
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
records = context.records
image_loader = context.image_loader
print(f"{len(records)} MIRA-Bench annotations loaded from {ANNOTATION_FILE}")
print("Fields:", sorted(records[0]))

In [ ]:
SAMPLE_INDEX = 0
record = records[SAMPLE_INDEX]
sample = load_sample(record, benchmark=BENCHMARK, image_loader=image_loader, prompt_mode=PROMPT_MODE)
if not sample["instruction"]:
    raise ValueError("MIRA-Bench record has no usable original instruction.")
select_reference_prompts(sample)
print(sample["sample_id"], sample["instruction"])
print("Text mode uses natural-language reference descriptions; no visual selectors are supplied.")

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(annotate_sample(sample))
ax.set_title("MIRA-Bench source (text-only prompt)")
ax.axis("off")
plt.show()

In [ ]:
from framework.tools import image_editing_tools
from evaluation.notebook import create_agent_model

llm = create_agent_model()
tools = image_editing_tools()
print("MIRA-Bench agent ready:", llm.model_id)

In [ ]:
SAMPLE_LIMIT = None  # Set to a positive number (e.g. 1) for a smoke test.
if SAMPLE_LIMIT is not None and SAMPLE_LIMIT < 1:
    raise ValueError("SAMPLE_LIMIT must be positive or None.")
SAMPLE_COUNT = len(records) if SAMPLE_LIMIT is None else min(SAMPLE_LIMIT, len(records))
SAMPLE_INDICES = list(range(SAMPLE_COUNT))
RESUME_LAST_RUN = False  # True continues the latest run and skips completed samples.
from evaluation.notebook import resolve_run_directory

RUN_DIR = resolve_run_directory(OUTPUT_ROOT) if RESUME_LAST_RUN else create_run_directory(OUTPUT_ROOT)
benchmark_results = run_benchmark(
    records,
    SAMPLE_INDICES,
    benchmark=BENCHMARK,
    prompt_mode=PROMPT_MODE,
    output_dir=RUN_DIR,
    image_loader=image_loader,
    model=llm,
    tools=tools,
    resume=RESUME_LAST_RUN,
    display_fn=None,
)
print("Run directory:", RUN_DIR)
print(f"Processed {len(benchmark_results)} of {SAMPLE_COUNT} MIRA-Bench samples.")

In [ ]:
completed = [result for result in benchmark_results if result["status"] == "completed"]
failed = [result for result in benchmark_results if result["status"] == "failed"]
print(f"MIRA-Bench samples: {len(benchmark_results)} | completed: {len(completed)} | failed: {len(failed)}")
if benchmark_results:
    print(f"Mean runtime: {sum(result['elapsed_seconds'] for result in benchmark_results) / len(benchmark_results):.2f}s")
print("Review saved outputs and trajectories; runtime/status are not image-quality scores.")